In [ ]:
import torch

inputs = torch.tensor([[0.43, 0.15, 0.89] , # Your
                       [0.55, 0.87, 0.66],  # journey
                       [0.57, 0.85, 0.64],  # starts
                       [0.22, 0.58, 0.33],  # with
                       [0.77, 0.25, 0.10],  # one
                       [0.05, 0.80, 0.55]]) # step

query = inputs[1]
attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(query , x_i)
print("Attention scores for the second token (journey):", attn_scores_2)

attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()
print("Attention weights for the second token (journey):", attn_weights_2_tmp)
print("Sum of attention weights (should be 1):", attn_weights_2_tmp.sum())


def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0) # Formula = exp(x_i) / sum(exp(x_j)) for all j

attn_weights_2 = softmax_naive(attn_scores_2)
print("Attention weights for the second token (journey) after softmax:", attn_weights_2)
print("Sum of attention weights after softmax (should be 1):", attn_weights_2.sum())

attn_weights_3 = torch.softmax(attn_scores_2, dim=0) # Using PyTorch's built-in softmax function
print("Attention weights for the second token (journey) after PyTorch softmax:", attn_weights_3)
print("Sum of attention weights after PyTorch softmax (should be 1):", attn_weights_3.sum())

In [ ]:
query = inputs[1]
context_vector_2 = torch.zeros(inputs.shape[1]) # Initialize context vector with zeros
for i, x_i in enumerate(inputs):
    context_vector_2 += attn_weights_2[i] * x_i # Weighted sum of input vectors based on attention weights
print("Context vector for the second token (journey):", context_vector_2)
print("Actual input vector for the second token (journey):", query)

In [ ]:
print("Actual input vectors for all tokens:\n", inputs)

attn_scores = inputs @ inputs.T # Compute attention scores for all tokens using matrix multiplication
print("Attention scores for all tokens:\n", attn_scores)
attn_weights = torch.softmax(attn_scores, dim=1) # Apply softmax to get attention weights for all tokens
print("Attention weights for all tokens:\n", attn_weights)

context_vectors = attn_weights @ inputs # Compute context vectors for all tokens using matrix multiplication
print("Context vectors for all tokens:\n", context_vectors)

In [ ]:
x_2 = inputs[1]
xd_in = inputs.shape[1]
xd_out = 2

torch.manual_seed(123) # Set random seed for reproducibility
W_query = torch.nn.Parameter(torch.rand(xd_in, xd_out) , requires_grad=False) # Learnable weight matrix for query
W_key = torch.nn.Parameter(torch.rand(xd_in, xd_out), requires_grad=False)   # Learnable weight matrix for key
W_value = torch.nn.Parameter(torch.rand(xd_in, xd_out), requires_grad=False) # Learnable weight matrix for value

print("Input vector for the second token (journey):", x_2)
print("Query weight matrix:\n", W_query)
print("Key weight matrix:\n", W_key)
print("Value weight matrix:\n", W_value)


In [ ]:
query_2 = x_2 @ W_query # Compute query vector for the second token
key_2 = x_2 @ W_key     # Compute key vector for the second token
value_2 = x_2 @ W_value # Compute value vector for the second token

print("Query vector for the second token (journey):", query_2)
print("Key vector for the second token (journey):", key_2)
print("Value vector for the second token (journey):", value_2)

In [ ]:
keys = inputs @ W_key   # Compute key vectors for all tokens
queries = inputs @ W_query # Compute query vectors for all tokens
values = inputs @ W_value # Compute value vectors for all tokens
print("Key vectors for all tokens:\n", keys)
print("Query vectors for all tokens:\n", queries)
print("Value vectors for all tokens:\n", values)

In [ ]:
keys_2 = keys[1] # Key vector for the second token
attn_scores_22= queries[1] @ keys_2 # Compute attention scores for the second token
print("Attention scores for the second token (journey) using keys:\n", attn_scores_22)

In [ ]:
attn_scores = query_2 @ keys.T # Compute attention scores for the second token against all keys
print("Attention scores for the second token (journey) against all keys:\n", attn_scores)

attn_weights = torch.softmax(attn_scores / (keys.shape[1] ** 0.5), dim=0) # Compute attention weights using softmax
print("Attention weights for the second token (journey) against all keys:\n", attn_weights)

context_vectors = attn_weights @ values # Compute context vector for the second token using attention weights and value vectors
print("Context vector for the second token (journey) using attention weights and value vectors:\ n", context_vectors)

In [ ]:
import torch

# Shape: [3, 2] -> (3 tokens, 2 features)
matrix_2d = torch.tensor([[11, 12],
                          [21, 22],
                          [31, 32]])

# Using .T or transpose(-2, -1) does the exact same thing here
transposed_2d = matrix_2d.transpose(-2, -1)

print("Original 2D Shape:", matrix_2d.shape)      # torch.Size([3, 2])
print("Transposed 2D Shape:", transposed_2d.shape)  # torch.Size([2, 3])
print("\nTransposed Matrix:\n", transposed_2d)
# Output:
# tensor([[11, 21, 31],
#         [12, 22, 32]])

### Example 2: The 3D Batch (Why .T Crashes)

Now, imagine we are processing a batch of 2 separate sentences at the exact same time. Each sentence still has 3 tokens, and each token has 2 features.

Our tensor shape is now [2, 3, 2] -> [Batch_Size, Seq_Len, Feature_Dim].

If you try to run matrix_3d.T, PyTorch will throw a runtime error because .T expects exactly a 2D matrix. It doesn't know what to do with the batch dimension sitting at index 0.

Here is how transpose(-2, -1) handles it cleanly:

In [ ]:
# Shape: [2, 3, 2] -> (Batch of 2 sentences, 3 tokens each, 2 features each)
matrix_3d = torch.tensor([
    [[11, 12], [21, 22], [31, 32]],  # Sentence 1
    [[51, 52], [61, 62], [71, 72]]   # Sentence 2
])

# Fix: Only swap dimension -2 (3) and dimension -1 (2)
transposed_3d = matrix_3d.transpose(-2, -1)

print("Original 3D Shape:", matrix_3d.shape)      # torch.Size([2, 3, 2])
print("Transposed 3D Shape:", transposed_3d.shape)  # torch.Size([2, 2, 3])

print("\nSentence 1 Transposed:\n", transposed_3d[0])
# Output:
# tensor([[11, 21, 31],
#         [12, 22, 32]])

print("\nSentence 2 Transposed:\n", transposed_3d[1])
# Output:
# tensor([[51, 61, 71],
#         [52, 62, 72]])

try:
    invalid_transpose = matrix_3d.T # This will raise an error because it tries to swap batch and token dimensions
except RuntimeError as e:
    print("Error during invalid transpose:", e)

In [ ]:
import torch.nn as nn
    
class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out)) 
        self.W_key = nn.Parameter(torch.rand(d_in, d_out))   
        self.W_value = nn.Parameter(torch.rand(d_in, d_out)) 

    def forward(self, x):
        keys = x @ self.W_key   
        queries = x @ self.W_query 
        values = x @ self.W_value 
        attn_scores = queries @ keys.transpose(-2, -1)
        attn_weights = torch.softmax(attn_scores / (keys.shape[1] ** 0.5), dim=-1) 
        context_vectors = attn_weights @ values 
        return context_vectors


In [ ]:
print(inputs.shape , W_query.shape , W_key.shape , W_value.shape)

In [ ]:
d_in , d_out = inputs.shape[1], 2

torch.manual_seed(123) # Set random seed for reproducibility
sa_v1 = SelfAttention_v1(d_in , d_out)
print("Self-Attention module initialized with random weights:\n", sa_v1(inputs))

In [ ]:
class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias) # Learnable linear layer for query
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias) # Learnable linear layer for value

    def forward(self, x):
        keys = self.W_key(x)   # Compute key vectors using linear layer
        queries = self.W_query(x) # Compute query vectors using linear layer
        values = self.W_value(x) # Compute value vectors using linear layer
        attn_scores = queries @ keys.transpose(-2, -1) # Compute attention scores
        attn_weights = torch.softmax(attn_scores / (keys.shape[1] ** 0.5), dim=-1) # Compute attention weights
        context_vectors = attn_weights @ values # Compute context vectors
        return context_vectors


In [ ]:
d_in , d_out = inputs.shape[1], 2

torch.manual_seed(123) # Set random seed for reproducibility
sa_v2 = SelfAttention_v2(d_in , d_out)
print("Self-Attention module initialized with random weights:\n", sa_v1(inputs))

In [ ]:
queries = sa_v2.W_query(inputs) # Compute query vectors using the linear layer from SelfAttention_v2
keys = sa_v2.W_key(inputs)   # Compute key vectors using the linear layer from SelfAttention_v2
values = sa_v2.W_value(inputs) # Compute value vectors using the linear layer from SelfAttention_v2

attn_scores = queries @ keys.T # Compute attention scores using the query and key vectors
print("Attention scores using the linear layers from SelfAttention_v1:\n", attn_scores)
attn_weights = torch.softmax(attn_scores / (keys.shape[1] ** 0.5), dim=0) # Compute attention weights using softmax
print("Attention weights using the linear layers from SelfAttention_v1:\n", attn_weights)
context_vectors = attn_weights @ values # Compute context vectors
print("Context vectors using the linear layers from SelfAttention_v1:\n", context_vectors)

### Now, let's step smoothly into Masked (Causal) Attention using the code setup we just validated.

Why Masked Attention?
In an autoregressive LLM (like GPT), the model is trained to predict the next word. If we allow Token 2 ("journey") to look forward and see Token 3 ("starts"), the model will just copy that word. It cheats.

To train it properly, we must build a wall so that each token can only see itself and the tokens that came before it.

Step 1: Creating the Causal Mask Matrix
To block the future, we use a lower triangular matrix. Let's create it for a sequence of 6 tokens.

In [ ]:
context_length = attn_scores.shape[0]
mask_simple_lower_triangular = torch.tril(torch.ones(context_length, context_length))
print("Simple lower triangular mask:\n", mask_simple_lower_triangular)

mask_simple_upper_triangular = torch.triu(torch.ones(context_length, context_length)).bool()
print("Simple upper triangular mask:\n", mask_simple_upper_triangular)
print("Simple upper triangular mask (int):\n", mask_simple_upper_triangular.int())

In [ ]:
a = torch.tensor([[1,2,3], [4,5,6] , [7,8,9], [10,11,12]])
print(a)
print(a.shape , a.shape[0], a.shape[1] , a.shape[-2], a.shape[-1])
print(a.transpose(-2, -1))
print(a)

print("Sum along rows: ", a.sum(dim = 0)) # Sum along the first dimension (rows) - This will give us the sum of each column
print("Sum along columns: ", a.sum(dim = 1)) # Sum along the second dimension (columns) - This will give us the sum of each row
print("Sum along last dimension: ", a.sum(dim=-1)) # Sum along the last dimension (columns) - This will give us the sum of each row
print("Sum along second to last dimension: ", a.sum(dim = -2 )) # Sum along the second to last dimension (rows) - This will give us the sum of each column

In [ ]:
### Simple implmenttaion

attn_weights_clone = attn_weights.clone() # Create a copy of attention weights to apply masking
masked_attn_weights = attn_weights_clone * mask_simple_lower_triangular # Apply lower triangular mask to attention weights
print("Masked attention weights (lower triangular):\n", masked_attn_weights)

row_sums_masked = masked_attn_weights.sum(dim=-1, keepdim=True)
normalized_masked_attn_weights = masked_attn_weights / row_sums_masked # Normalize masked attention weights
print("Normalized masked attention weights (lower triangular):\n", normalized_masked_attn_weights)

In [ ]:
attn_weights_cloned = attn_weights.clone()
mask = torch.triu(torch.ones(context_length, context_length) , diagonal=1).bool() # Create a boolean lower triangular mask
print("Boolean upper triangular mask:\n", mask)

attn_weights_cloned_v2 = attn_weights_cloned.masked_fill(mask, float('-inf')) # Fill masked positions with -inf to effectively zero them out after softmax
print("Attention weights after applying masked_fill with -inf:\n", attn_weights_cloned_v2)

normalized_masked_attn_weights_v2 = torch.softmax(attn_weights_cloned_v2, dim=-1) # Apply softmax to get normalized masked attention weights
print("Normalized masked attention weights (upper triangular) using masked_fill and softmax:\n", normalized_masked_attn_weights_v2)

## Causal Masking (Preventing Future Token Attention)

**Key Concepts:**
1. **Order matters**: Mask scores BEFORE softmax, not after
2. **dim=1**: Each row (query) attends to all keys (columns), so softmax across dim=1
3. **Use -inf**: Replace masked positions with -inf so exp(-inf) = 0 after softmax

**Why dim=1?**
- Attention matrix shape: `[num_queries, num_keys]`
- Each **row** = how one query attends to all keys
- Each row should sum to 1 → softmax across dim=1 (across columns)

In [ ]:
torch.manual_seed(123) # Set random seed for reproducibility

dropout = torch.nn.Dropout(p=0.5) # Create a dropout layer with 50% dropout rate
example = torch.rand(5,5) # Example input tensor
print("Example input tensor:\n", example)
print("Example input tensor after applying dropout:\n", dropout(example))